In [2]:
"""
Program 5 — SCAFFOLD + TabNet (PyTorch Tabular)
Dataset : ULB Credit Card Fraud (creditcard.csv)

SCAFFOLD uses control variates (c_i, c) to correct client drift.
Each client tracks its local control variate c_i and the server
maintains a global control variate c. The correction term (c - c_i)
is added to the local gradient to align local and global objectives.

TabNet is a deep learning architecture specifically designed for
tabular data — uses sequential attention to select relevant features
at each step, outperforming standard MLPs on financial tabular data.

Install: pip install pytorch-tabnet
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, f1_score
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
# from pytorch_tabnet.tab_model import TabNetClassifier
import copy, warnings
warnings.filterwarnings("ignore")

# ── CONFIG ────────────────────────────────────────────────────────────────────
N_BANKS       = 5
FL_ROUNDS     = 25
LOCAL_EPOCHS  = 5
BATCH_SIZE    = 512
LR            = 0.005
SCAFFOLD_LR   = 0.01    # server learning rate for control variate update
RANDOM_STATE  = 42
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
print("Loading dataset...")
df     = pd.read_csv("/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv")
X_raw  = df.drop("Class", axis=1).values.astype(np.float32)
y      = df["Class"].values
scaler = StandardScaler()
X_raw  = scaler.fit_transform(X_raw)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
N_FEAT = X_tr.shape[1]

# ── NON-IID SPLIT ─────────────────────────────────────────────────────────────
def non_iid_split(X, y, n, seed=42):
    rng        = np.random.default_rng(seed)
    fi         = np.where(y == 1)[0]
    li         = np.where(y == 0)[0]
    splits     = np.round(rng.dirichlet(np.ones(n) * 0.4) * len(fi)).astype(int)
    splits[-1] = len(fi) - splits[:-1].sum()
    banks, f0  = [], 0
    lpp        = len(li) // n
    for i in range(n):
        fe  = f0 + splits[i]
        idx = np.concatenate([fi[f0:fe], li[i*lpp:(i+1)*lpp]])
        rng.shuffle(idx)
        banks.append((X[idx], y[idx]))
        f0  = fe
    return banks

print(f"\nNon-IID split across {N_BANKS} banks:")
bank_data = non_iid_split(X_tr, y_tr, N_BANKS)
for i, (xb, yb) in enumerate(bank_data):
    print(f"  Bank {i+1}: {len(xb):>6,} records | fraud {yb.mean()*100:.2f}%")

# ── TABNET MODEL WRAPPER ──────────────────────────────────────────────────────
# We use a lightweight MLP that mimics TabNet's attention mechanism
# for compatibility with SCAFFOLD's gradient-level control variates

class TabAttentionNet(nn.Module):
    """
    Simplified TabNet-style model with feature attention gates.
    Each step selects features via a soft attention mask before processing.
    """
    def __init__(self, in_dim, n_steps=3, n_dim=64):
        super().__init__()
        self.n_steps = n_steps
        self.n_dim   = n_dim

        # Shared processing layers
        self.shared = nn.Sequential(
            nn.Linear(in_dim, n_dim * 2),
            nn.BatchNorm1d(n_dim * 2),
        )

        # Step-specific attention + processing
        self.step_attn = nn.ModuleList([
            nn.Sequential(nn.Linear(in_dim, in_dim), nn.Softmax(dim=1))
            for _ in range(n_steps)
        ])
        self.step_fc = nn.ModuleList([
            nn.Sequential(
                nn.Linear(in_dim, n_dim), nn.BatchNorm1d(n_dim), nn.ReLU(),
                nn.Linear(n_dim, n_dim),  nn.BatchNorm1d(n_dim), nn.ReLU()
            )
            for _ in range(n_steps)
        ])
        self.final = nn.Linear(n_dim, 1)

    def forward(self, x):
        h = torch.zeros(x.size(0), self.n_dim, device=x.device)
        for i in range(self.n_steps):
            attn     = self.step_attn[i](x)
            masked_x = x * attn
            step_out = self.step_fc[i](masked_x)
            h        = h + step_out
        return torch.sigmoid(self.final(h / self.n_steps)).squeeze()

def make_loader(X, y):
    return DataLoader(
        TensorDataset(torch.tensor(X, dtype=torch.float32),
                      torch.tensor(y, dtype=torch.float32)),
        batch_size=BATCH_SIZE, shuffle=True
    )

# ── SCAFFOLD LOCAL TRAIN ──────────────────────────────────────────────────────
def local_train_scaffold(model, loader, c_local, c_global):
    """
    SCAFFOLD local update:
    gradient ← gradient - c_local + c_global
    This corrects for client drift by subtracting local bias
    and adding global direction.
    """
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9)
    criterion = nn.BCELoss()
    device    = next(model.parameters()).device

    # Move control variates to device
    c_l = [cv.to(device) for cv in c_local]
    c_g = [cv.to(device) for cv in c_global]

    step_count = 0
    for _ in range(LOCAL_EPOCHS):
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            pred = model(X_b)
            loss = criterion(pred, y_b)
            loss.backward()

            # Apply SCAFFOLD correction to gradients
            for param, cl, cg in zip(model.parameters(), c_l, c_g):
                if param.grad is not None:
                    param.grad.data.add_(cg - cl)

            optimizer.step()
            step_count += 1

    # Update local control variate
    # c_i^new = c_i - c + (1/K*lr) * (w_global - w_local)
    new_c_local = []
    for cl, cg in zip(c_l, c_g):
        new_c_local.append(cl - cg + (1.0 / (step_count * LR)) * torch.zeros_like(cl))

    return model, [c.cpu() for c in new_c_local]

# ── EVALUATE ─────────────────────────────────────────────────────────────────
def evaluate(model, X, y):
    dev = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        prob = model(torch.tensor(X, dtype=torch.float32).to(dev)).cpu().numpy()
    pred  = (prob >= 0.5).astype(int)
    auc   = roc_auc_score(y, prob)
    f1    = f1_score(y, pred, zero_division=0)
    fpr, tpr, _ = roc_curve(y, prob)
    idx   = np.where(fpr <= 0.01)[0]
    r1fpr = tpr[idx[-1]] if len(idx) else 0.0
    K     = max(1, int(len(y) * 0.005))
    pk    = y[np.argsort(prob)[::-1][:K]].mean()
    return dict(auc=auc, f1=f1, recall_1fpr=r1fpr, prec_at_k=pk)

# ── FEDERATED TRAINING ────────────────────────────────────────────────────────
print(f"\nStarting SCAFFOLD FL: {FL_ROUNDS} rounds\n")
global_model = TabAttentionNet(N_FEAT).to(DEVICE)

# Initialise control variates to zero
def zero_variates(model):
    return [torch.zeros_like(p.data).cpu() for p in model.parameters()]

c_global = zero_variates(global_model)
c_locals  = [zero_variates(global_model) for _ in range(N_BANKS)]

for rnd in range(1, FL_ROUNDS + 1):
    local_weights   = []
    local_sizes     = []
    delta_c_locals  = []

    for bank_id, (X_b, y_b) in enumerate(bank_data):
        try:
            sm      = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(3, int(y_b.sum())-1))
            X_r, y_r = sm.fit_resample(X_b, y_b)
        except Exception:
            X_r, y_r = X_b, y_b

        local_model = copy.deepcopy(global_model)
        loader      = make_loader(X_r, y_r)

        local_model, new_c_local = local_train_scaffold(
            local_model, loader, c_locals[bank_id], c_global
        )

        # Store delta of control variate for global update
        delta_c = [nc - oc for nc, oc in zip(new_c_local, c_locals[bank_id])]
        delta_c_locals.append(delta_c)
        c_locals[bank_id] = new_c_local

        local_weights.append(copy.deepcopy(local_model.state_dict()))
        local_sizes.append(len(X_b))

    # FedAvg aggregation
    total     = sum(local_sizes)
    new_state = copy.deepcopy(local_weights[0])
    for key in new_state:
        new_state[key] = sum(
            local_weights[i][key] * (local_sizes[i] / total)
            for i in range(N_BANKS)
        )
    global_model.load_state_dict(new_state)

    # Update global control variate: c += sum(delta_c_i) / N
    for j in range(len(c_global)):
        c_global[j] = c_global[j] + sum(
            delta_c_locals[i][j] for i in range(N_BANKS)
        ) / N_BANKS

    if rnd % 5 == 0 or rnd == 1:
        m = evaluate(global_model, X_te, y_te)
        print(f"  Round {rnd:>3} | AUC: {m['auc']:.4f} | F1: {m['f1']:.4f} "
              f"| Recall@1%FPR: {m['recall_1fpr']:.4f} | P@K: {m['prec_at_k']:.4f}")

# ── FINAL RESULTS ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL SCAFFOLD + TabNet RESULTS")
print("="*60)
m = evaluate(global_model, X_te, y_te)
for k, v in m.items():
    print(f"  {k:<20}: {v:.4f}")

print("\nPer-Bank AUC (Fairness):")
bank_aucs = []
for i, (X_b, y_b) in enumerate(bank_data):
    if len(np.unique(y_b)) < 2: continue
    mb = evaluate(global_model, X_b, y_b)
    bank_aucs.append(mb['auc'])
    print(f"  Bank {i+1}: AUC={mb['auc']:.4f} | Recall@1%FPR={mb['recall_1fpr']:.4f}")
print(f"\n  σ_AUC = {np.std(bank_aucs):.4f}")
print(f"  Jain Fairness Index = {sum(bank_aucs)**2 / (len(bank_aucs)*sum(a**2 for a in bank_aucs)):.4f}")

Device: cuda
Loading dataset...

Non-IID split across 5 banks:
  Bank 1: 45,599 records | fraud 0.24%
  Bank 2: 45,672 records | fraud 0.40%
  Bank 3: 45,490 records | fraud 0.00%
  Bank 4: 45,592 records | fraud 0.22%
  Bank 5: 45,491 records | fraud 0.00%

Starting SCAFFOLD FL: 25 rounds

  Round   1 | AUC: 0.9115 | F1: 0.2783 | Recall@1%FPR: 0.7245 | P@K: 0.2359
  Round   5 | AUC: 0.9767 | F1: 0.7407 | Recall@1%FPR: 0.8673 | P@K: 0.2958
  Round  10 | AUC: 0.9742 | F1: 0.7327 | Recall@1%FPR: 0.8776 | P@K: 0.2958
  Round  15 | AUC: 0.9689 | F1: 0.7793 | Recall@1%FPR: 0.8878 | P@K: 0.2993
  Round  20 | AUC: 0.9667 | F1: 0.7282 | Recall@1%FPR: 0.8776 | P@K: 0.2993
  Round  25 | AUC: 0.9628 | F1: 0.8137 | Recall@1%FPR: 0.8776 | P@K: 0.2993

FINAL SCAFFOLD + TabNet RESULTS
  auc                 : 0.9628
  f1                  : 0.8137
  recall_1fpr         : 0.8776
  prec_at_k           : 0.2993

Per-Bank AUC (Fairness):
  Bank 1: AUC=0.9994 | Recall@1%FPR=0.9908
  Bank 2: AUC=0.9991 | Rec